
<div class="problem-banner">
<strong>Problema:</strong> un sistema recibe lotes de imágenes en escala de
grises producidas por sensores de control de calidad. Antes de entrenar un
clasificador debemos representar el lote, normalizar cada imagen, mover los
datos al dispositivo disponible y comprobar que las operaciones conservan las
dimensiones esperadas.
</div>

## Por qué comenzar por los tensores

En deep learning, casi toda la información termina representada como un
**tensor**: una colección rectangular de valores con una forma, un tipo y un
dispositivo. Las imágenes, los parámetros de una red, las predicciones y los
gradientes comparten esta representación.

Una expresión breve como

```python
loss = criterion(model(images), labels)
loss.backward()
```

oculta varias operaciones: un lote entra al modelo, PyTorch construye un grafo
computacional, la pérdida resume el error y `backward()` calcula cómo cambiar
cada parámetro. Para razonar sobre ese proceso primero debemos poder responder:

- ¿qué representa cada eje del tensor?;
- ¿qué tipo numérico contiene?;
- ¿en qué dispositivo reside?;
- ¿qué operación cambia su forma?;
- ¿qué valores participan en el cálculo de gradientes?

::: {.callout-note title="Objetivos de aprendizaje"}
Al terminar este capítulo podrás:

- crear tensores a partir de datos y describir `shape`, `dtype` y `device`;
- seleccionar, reorganizar y combinar dimensiones sin confundir observaciones
  con características;
- explicar broadcasting y distinguir multiplicación elemento a elemento de
  multiplicación matricial;
- normalizar un lote de imágenes respetando sus ejes;
- seleccionar CPU, CUDA o MPS y mover datos de forma coherente;
- construir un `Dataset` y recorrerlo mediante un `DataLoader`;
- usar autograd para calcular un gradiente sencillo; y
- reconocer errores comunes de dimensiones, tipos y dispositivos.
:::

## Preparar el entorno

El notebook está diseñado para PyTorch 2.4 o posterior. En Google Colab,
PyTorch suele estar instalado; la siguiente celda permite comprobar el entorno.

In [ ]:
import platform

import numpy as np
import torch

SEED = 42
torch.manual_seed(SEED)

print(f"Python: {platform.python_version()}")
print(f"PyTorch: {torch.__version__}")

La selección de dispositivo debe ocurrir en un solo lugar. CUDA corresponde a
GPU NVIDIA; MPS permite usar GPU en equipos Apple compatibles.

In [ ]:
def select_device():
    if torch.cuda.is_available():
        return torch.device("cuda")
    if torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")


device = select_device()
print(f"Dispositivo seleccionado: {device}")

::: {.callout-important title="Una regla operacional"}
Los datos de entrada y los parámetros del modelo deben estar en el mismo
dispositivo. Mover solo uno de ellos producirá un error en la primera operación
que los combine.
:::

## Del arreglo al tensor

Representaremos inicialmente cuatro imágenes de un sensor. Cada imagen tiene
un canal, cuatro filas y cinco columnas. La forma del lote es, por tanto,
$(N,C,H,W)=(4,1,4,5)$.

In [ ]:
image_values = [
    [
        [
            [12, 14, 15, 13, 11],
            [13, 16, 18, 17, 12],
            [11, 15, 19, 16, 10],
            [10, 12, 14, 13, 9],
        ]
    ],
    [
        [
            [30, 31, 29, 28, 30],
            [31, 35, 37, 34, 29],
            [29, 36, 42, 35, 28],
            [27, 29, 31, 30, 26],
        ]
    ],
    [
        [
            [8, 9, 8, 10, 9],
            [9, 11, 12, 11, 8],
            [8, 10, 13, 10, 7],
            [7, 8, 9, 8, 6],
        ]
    ],
    [
        [
            [20, 22, 21, 19, 18],
            [21, 25, 28, 24, 20],
            [19, 26, 32, 25, 18],
            [17, 20, 22, 20, 16],
        ]
    ],
]

images = torch.tensor(image_values, dtype=torch.float32)
labels = torch.tensor([0, 1, 0, 1], dtype=torch.int64)

print("Forma de imágenes:", images.shape)
print("Forma de etiquetas:", labels.shape)
print("Tipo de imágenes:", images.dtype)
print("Tipo de etiquetas:", labels.dtype)

La forma no es una anotación decorativa. Es parte del significado:

| Eje | Tamaño | Interpretación |
|---|---:|---|
| $N$ | 4 | imágenes en el lote |
| $C$ | 1 | canales por imagen |
| $H$ | 4 | filas |
| $W$ | 5 | columnas |

Las imágenes usan `float32` porque las operaciones de una red y sus gradientes
requieren números de punto flotante. Las etiquetas usan `int64` porque muchas
funciones de clasificación esperan índices enteros de clase.

### Propiedades que siempre deben inspeccionarse

In [ ]:
tensor_audit = {
    "shape": tuple(images.shape),
    "ndim": images.ndim,
    "numel": images.numel(),
    "dtype": str(images.dtype),
    "device": str(images.device),
    "requires_grad": images.requires_grad,
}
tensor_audit

`numel()` cuenta escalares. Aquí debe coincidir con
$4\times1\times4\times5=80$. Esta igualdad es útil para auditar operaciones de
reorganización.

### NumPy y memoria compartida

`torch.from_numpy()` puede compartir memoria con un arreglo de NumPy en CPU.
Modificar uno puede modificar el otro, una propiedad eficiente pero fácil de
pasar por alto.

In [ ]:
numpy_array = np.array([1.0, 2.0, 3.0], dtype=np.float32)
shared_tensor = torch.from_numpy(numpy_array)

numpy_array[0] = 99.0
print(shared_tensor)

Cuando se necesita independencia explícita se utiliza una copia:

In [ ]:
independent_tensor = torch.tensor(numpy_array)
numpy_array[1] = -10.0
print("NumPy:", numpy_array)
print("Tensor independiente:", independent_tensor)

## Seleccionar observaciones y regiones

La indexación conserva la lógica de NumPy. El siguiente acceso selecciona la
primera imagen, su único canal y todas las posiciones espaciales.

In [ ]:
first_image = images[0, 0, :, :]
print(first_image)
print(first_image.shape)

Eliminar el eje de canal produce una matriz $(H,W)$. En cambio, el corte
`images[0:1]` conserva el eje del lote:

In [ ]:
one_image_batch = images[0:1]
print("Imagen sin ejes de lote y canal:", first_image.shape)
print("Lote de una imagen:", one_image_batch.shape)

Esta diferencia importa porque una capa convolucional normalmente espera
cuatro dimensiones. Una selección que elimina accidentalmente el eje del lote
puede romper el modelo.

Podemos extraer una región central de todas las imágenes sin iterar:

In [ ]:
central_regions = images[:, :, 1:3, 1:4]
print(central_regions.shape)
print(central_regions)

## Cambiar la forma sin cambiar los datos

Una red densa recibe una matriz con una fila por observación. Para convertir
cada imagen de $1\times4\times5$ en 20 características usamos `flatten`:

In [ ]:
flat_images = images.flatten(start_dim=1)
print(flat_images.shape)

assert flat_images.shape == (4, 20)
assert flat_images.numel() == images.numel()

`reshape` permite expresar la misma intención cuando conocemos la forma de
destino. El valor `-1` pide a PyTorch inferir una dimensión.

In [ ]:
flat_with_reshape = images.reshape(images.shape[0], -1)
print(torch.equal(flat_images, flat_with_reshape))

`permute` no aplana: cambia el orden de los ejes. Esto es necesario al
intercambiar datos entre bibliotecas que usan $(N,H,W,C)$ y PyTorch, que usa
$(N,C,H,W)$.

In [ ]:
channels_last = images.permute(0, 2, 3, 1)
print("PyTorch NCHW:", images.shape)
print("Canales al final NHWC:", channels_last.shape)

::: {.callout-warning title="No intercambiar reshape y permute"}
`reshape` reorganiza cómo se agrupan los valores; `permute` cambia qué eje
representa cada posición. Ambos pueden producir un tensor con el mismo número
de elementos, pero no necesariamente con el mismo significado.
:::

## Operaciones vectorizadas

La media de intensidad por imagen puede obtenerse en una operación. Debemos
reducir canal, altura y anchura, pero conservar el eje del lote.

In [ ]:
mean_per_image = images.mean(dim=(1, 2, 3))
max_per_image = images.amax(dim=(1, 2, 3))

summary = torch.stack([mean_per_image, max_per_image], dim=1)
print(summary)
print("Forma del resumen:", summary.shape)

La reducción produce dos características por imagen. `stack` crea un eje nuevo;
`cat` concatena sobre un eje existente.

In [ ]:
first_half = images[:2]
second_half = images[2:]
reconstructed_batch = torch.cat([first_half, second_half], dim=0)

assert torch.equal(images, reconstructed_batch)

### Broadcasting

Queremos normalizar cada imagen con su propia media y desviación estándar:

$$
z_{nchw}=\frac{x_{nchw}-\mu_n}{\sigma_n+\varepsilon}.
$$

Las estadísticas deben conservar dimensiones unitarias para que PyTorch pueda
expandirlas virtualmente sobre canal, altura y anchura.

In [ ]:
means = images.mean(dim=(1, 2, 3), keepdim=True)
stds = images.std(dim=(1, 2, 3), keepdim=True)
normalized_images = (images - means) / (stds + 1e-6)

print("Imágenes:", images.shape)
print("Medias:", means.shape)
print("Normalizadas:", normalized_images.shape)
print("Medias después de normalizar:", normalized_images.mean(dim=(1, 2, 3)))

Broadcasting no copia físicamente la media 20 veces. PyTorch alinea las
dimensiones desde la derecha y permite combinar tamaños iguales o tamaños uno.

::: {.callout-tip title="Auditar broadcasting"}
Antes de una operación, escribe las formas una debajo de otra y alínealas a la
derecha. En este caso, `(4, 1, 4, 5)` y `(4, 1, 1, 1)` son compatibles.
:::

### Producto elemento a elemento y producto matricial

El operador `*` combina posiciones correspondientes. El operador `@` realiza
producto matricial.

In [ ]:
features = torch.tensor([
    [0.2, 1.0, -0.5],
    [0.7, 0.3, 0.1],
], dtype=torch.float32)
weights = torch.tensor([
    [0.4, -0.2],
    [0.1, 0.8],
    [-0.6, 0.3],
], dtype=torch.float32)

scores = features @ weights
print("Características:", features.shape)
print("Pesos:", weights.shape)
print("Scores:", scores.shape)
print(scores)

La regla dimensional es

$$
(B,D)@(D,K)=(B,K),
$$

donde $B$ es el lote, $D$ el número de características y $K$ el número de
salidas.

## Mover datos al dispositivo

Mover el lote es explícito. La variable original permanece en CPU; `to()`
devuelve un tensor en el dispositivo solicitado.

In [ ]:
images_device = images.to(device)
labels_device = labels.to(device)

print(images_device.device)
print(labels_device.device)

Para volver a NumPy se requiere CPU y un tensor desconectado del grafo:

In [ ]:
images_numpy = images_device.detach().cpu().numpy()
print(images_numpy.shape)

Copiar datos entre CPU y GPU tiene costo. En un ciclo de entrenamiento se
mueven mini-batches completos, no escalares individuales.

## Autograd: aprender significa diferenciar

Supongamos un modelo lineal muy pequeño para estimar un score a partir de las
dos características del resumen. Sus parámetros son un vector $\mathbf{w}$ y
un intercepto $b$:

$$
\widehat{y}_i = \mathbf{x}_i^\mathsf{T}\mathbf{w}+b.
$$

In [ ]:
X = summary
y = labels.to(torch.float32)

w = torch.zeros(2, requires_grad=True)
b = torch.zeros(1, requires_grad=True)

predictions = X @ w + b
loss = ((predictions - y) ** 2).mean()
loss.backward()

print("Pérdida:", loss.item())
print("Gradiente de w:", w.grad)
print("Gradiente de b:", b.grad)

`requires_grad=True` indica que las operaciones sobre esos tensores deben
registrarse. `backward()` aplica la regla de la cadena desde la pérdida escalar
hasta cada parámetro hoja.

Para el error cuadrático medio,

$$
\nabla_{\mathbf{w}}\mathcal{L}
=\frac{2}{N}\mathbf{X}^\mathsf{T}
(\mathbf{X}\mathbf{w}+b-\mathbf{y}).
$$

Podemos comprobar que el gradiente automático coincide con la expresión:

In [ ]:
manual_gradient = 2 / len(X) * X.T @ (X @ w.detach() + b.detach() - y)
print("Gradiente manual:", manual_gradient)
print("Coinciden:", torch.allclose(w.grad, manual_gradient))

Los gradientes se acumulan. Antes de calcular un nuevo gradiente debemos
reiniciarlos. Los optimizadores realizan este paso mediante
`optimizer.zero_grad()`.

In [ ]:
with torch.no_grad():
    learning_rate = 0.001
    w -= learning_rate * w.grad
    b -= learning_rate * b.grad

w.grad.zero_()
b.grad.zero_()

`torch.no_grad()` evita registrar la actualización como parte del grafo. En los
capítulos siguientes un optimizador reemplazará esta actualización manual.

## Organizar datos con Dataset y DataLoader

Los proyectos reales no entregan todos los datos en una lista escrita a mano.
PyTorch separa dos responsabilidades:

- `Dataset` define cómo obtener una observación y su etiqueta;
- `DataLoader` agrupa, mezcla y entrega mini-batches.

Para tensores que ya caben en memoria podemos usar `TensorDataset`.

In [ ]:
from torch.utils.data import DataLoader, TensorDataset

dataset = TensorDataset(normalized_images, labels)
loader = DataLoader(
    dataset,
    batch_size=2,
    shuffle=True,
    generator=torch.Generator().manual_seed(SEED),
)

for batch_id, (batch_images, batch_labels) in enumerate(loader):
    print(
        f"Batch {batch_id}: ",
        f"X={tuple(batch_images.shape)}, ",
        f"y={tuple(batch_labels.shape)}",
    )

`shuffle=True` altera el orden al comienzo de cada época. La semilla del
generador hace reproducible este ejemplo, aunque en un entrenamiento completo
también deben controlarse otras fuentes de aleatoriedad.

### Un ciclo mínimo de preparación

El patrón que reutilizaremos es:

In [ ]:
for batch_images, batch_labels in loader:
    batch_images = batch_images.to(device)
    batch_labels = batch_labels.to(device)

    assert batch_images.ndim == 4
    assert batch_images.shape[0] == batch_labels.shape[0]

Las aserciones convierten supuestos silenciosos en errores cercanos a su causa.
En proyectos posteriores también validaremos rangos, valores faltantes y
correspondencia de clases.

## Diagnóstico de errores frecuentes

### Ejes en el orden incorrecto

Una forma como `(32, 224, 224, 3)` puede parecer válida, pero una CNN de PyTorch
la interpretaría como 224 canales. La corrección es
`tensor.permute(0, 3, 1, 2)`.

### Etiquetas con tipo incorrecto

Para clasificación multiclase, `CrossEntropyLoss` espera normalmente índices
de clase `int64`, no valores `float32` ni vectores one-hot.

### Datos y modelo en dispositivos diferentes

El mensaje suele mencionar que se encontraron tensores en dos dispositivos.
Centralizar la selección y mover cada mini-batch reduce este riesgo.

### Gradientes acumulados accidentalmente

Dos llamadas a `backward()` suman gradientes si no se reinician. El ciclo
estándar es `zero_grad()`, forward, pérdida, `backward()` y `step()`.

### Eliminar el eje del lote

`images[0]` y `images[0:1]` no tienen la misma forma. El segundo conserva un
lote de tamaño uno.

## Solución integrada del problema

Ya podemos construir una función que recibe un lote, valida su estructura,
normaliza cada imagen y lo mueve al dispositivo.

In [ ]:
def prepare_image_batch(batch, target_device):
    if batch.ndim != 4:
        raise ValueError(
            "Se esperaba un tensor (N, C, H, W); "
            f"se recibió {tuple(batch.shape)}"
        )
    if not batch.is_floating_point():
        batch = batch.to(torch.float32)
    if not torch.isfinite(batch).all():
        raise ValueError("El lote contiene valores no finitos")

    means = batch.mean(dim=(1, 2, 3), keepdim=True)
    stds = batch.std(dim=(1, 2, 3), keepdim=True)
    normalized = (batch - means) / stds.clamp_min(1e-6)
    return normalized.to(target_device)


prepared_images = prepare_image_batch(images, device)

assert prepared_images.shape == images.shape
assert prepared_images.device.type == device.type
assert torch.allclose(
    prepared_images.mean(dim=(1, 2, 3)).cpu(),
    torch.zeros(len(images)),
    atol=1e-5,
)

print("Lote preparado:", prepared_images.shape, prepared_images.device)

La función no entrena todavía un clasificador. Resuelve una etapa previa y
crítica: entrega al modelo un lote con contrato explícito, tipo apropiado,
valores finitos, normalización por observación y dispositivo coherente.

## Qué hemos aprendido

- Un tensor se interpreta conjuntamente por sus valores, forma, tipo y
  dispositivo.
- En visión, PyTorch usa normalmente el orden $(N,C,H,W)$.
- La indexación puede conservar o eliminar ejes; esa diferencia afecta a las
  capas posteriores.
- `reshape`, `flatten` y `permute` resuelven problemas distintos.
- Broadcasting permite vectorizar operaciones sin copiar datos innecesarios.
- `*` opera elemento a elemento; `@` contrae dimensiones compatibles.
- Autograd registra operaciones y aplica la regla de la cadena desde una
  pérdida escalar.
- `Dataset` representa observaciones y `DataLoader` construye mini-batches.
- Las validaciones de forma, tipo, finitud y dispositivo deben ocurrir cerca de
  la entrada del sistema.

En el próximo capítulo estos elementos se usarán para aprender una relación a
partir de datos y construir un ciclo de entrenamiento completo.

## Ejercicios

1. Crea un tensor de diez imágenes RGB de $32\times32$. Explica cada eje y
   calcula el número total de escalares sin usar `numel()`; comprueba después el
   resultado con PyTorch.
2. Convierte un lote con forma `(N, H, W, C)` a `(N, C, H, W)` y verifica que un
   píxel específico conserva sus tres canales.
3. Calcula media y desviación estándar por canal para un lote RGB. Escribe las
   formas de los tensores antes de usar broadcasting.
4. Implementa una estandarización global y compárala con la normalización por
   imagen de este capítulo. Explica qué información elimina cada estrategia.
5. Construye una matriz de 20 características por imagen y multiplícala por una
   matriz de pesos para obtener tres scores por observación. Justifica todas las
   dimensiones.
6. Modifica el ejemplo de autograd para usar error absoluto medio. ¿Qué ocurre
   con el gradiente cuando una predicción coincide exactamente con la etiqueta?
7. Crea un `DataLoader` con cinco observaciones y `batch_size=2`. Registra el
   tamaño de cada mini-batch y explica el último resultado.
8. Extiende `prepare_image_batch()` para aceptar un rango esperado de valores y
   producir un error informativo cuando el lote lo incumpla.

## Reto

Construye un `Dataset` que genere imágenes sintéticas de $16\times16$ con dos
clases: imágenes con una franja vertical brillante e imágenes con una franja
horizontal. Visualiza ejemplos, crea particiones de entrenamiento y validación,
y entrega mini-batches normalizados. No entrenes todavía la red: documenta el
contrato de datos que deberá recibir el modelo del Capítulo 3.